In [0]:
%sql
USE CATALOG sofie_spark_dbt

In [0]:
%sql
USE SCHEMA `dbt-schema`

In [0]:
%sql
describe silver_dedu_ALL_Updates

In [0]:
%sql
SELECT *
from silver_dedu_ALL_Updates
-- where hours_diff < 1000 and hours_diff > 1

In the code below I will try to make a ML prediction  
of estimated time of service for an incoming order  
using the model **silver_dedu_ML** created in **dbt-core**.




In [0]:
from pyspark.sql.functions import col, log1p, expm1, concat_ws, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

# 1. Deal with rare Categorical values that may scew the data too much
def group_rare(df, colname, min_count=20):
    freq = df.groupBy(colname).count()
    rare = freq.filter(col("count") < min_count).select(colname).rdd.flatMap(lambda x: x).collect()
    return df.withColumn(colname, when(col(colname).isin(rare), "Other").otherwise(col(colname)))

categorical_cols = ["RegionId", "ProfessionId", "TaskCategoryId", "TaskTypeId", "TaskPriorityId"]

for c in categorical_cols:
    df = group_rare(df, c)

# 2. Interactive feature, to catch effect of combination
df = df.withColumn("Region_TaskType", concat_ws("_", col("RegionId"), col("TaskTypeId")))

categorical_cols.append("Region_TaskType")

# 3. Log-transform of target
df = df.withColumn("log_hours_diff", log1p(col("hours_diff")))

# 4. Train/test split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 5. Indexera Categorical features
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_Index", handleInvalid="keep")
    for c in categorical_cols
]

# 6. Assembler
assembler = VectorAssembler(
    inputCols=[f"{c}_Index" for c in categorical_cols],
    outputCol="features"
)

# 7. GBT Regressor
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="log_hours_diff",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=0.7,
    minInstancesPerNode=10,
    maxBins=64
)

# 8. Pipeline
pipeline = Pipeline(stages=indexers + [assembler, gbt])
model = pipeline.fit(train_df)

# 9. Prediktion + revert from LOG
predictions = model.transform(test_df)
predictions = predictions.withColumn("prediction_hours_diff", expm1(col("prediction")))

predictions.select("hours_diff", "prediction_hours_diff").show(100)
